# **Solving CVRP Problem Using Genetic Algorithm & Tabu Search**

This section loads CVRP data from Excel, computes the distance matrix, and defines a capacity-feasibility checker.

In [16]:
import numpy as np
import pandas as pd
from math import sqrt

# ----------------------------------------
# LOAD DATA (with print statements)
# ----------------------------------------

def load_cvrp_data(path, sheet="19MDVRP Problem Sets"):
    print("Loading CVRP data from file:", path)
    print("Using sheet:", sheet)

    df = pd.read_excel(path, sheet_name=sheet)

    print("\n--- Loaded DataFrame Head ---")
    print(df.head())

    print("\n--- DataFrame Columns ---")
    print(df.columns.tolist())

    # Expecting columns: X, Y, Demand
    print("\nExtracting columns ['X', 'Y', 'Demand'] ...")

    coords = df[['X', 'Y']].to_numpy()
    demands = df['Demand'].to_numpy()

    print("\nCoordinates Loaded:")
    print(coords[:10])  # first 10

    print("\nDemands Loaded:")
    print(demands[:10])

    print("\nData successfully loaded.\n")
    return coords, demands


# ----------------------------------------
# DISTANCE MATRIX (with print statements)
# ----------------------------------------

def compute_distance_matrix(coords):
    print("Computing distance matrix...")
    n = len(coords)
    print("Number of points:", n)

    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i, j] = sqrt((coords[i][0] - coords[j][0])**2 +
                              (coords[i][1] - coords[j][1])**2)

    print("Distance matrix computed. Sample:")
    print(dist[:5, :5])   # print a 5×5 block

    return dist


# ----------------------------------------
# CAPACITY CHECK (with print statements)
# ----------------------------------------

def is_capacity_feasible(route, demands, capacity):
    print("\nChecking route capacity...")
    print("Route:", route)
    print("Customer demands:", [demands[i] for i in route])
    print("Vehicle capacity:", capacity)

    total_load = sum(demands[i] for i in route)
    print("Total route load:", total_load)

    feasible = total_load <= capacity
    print("Is feasible?", feasible)

    return feasible


Checks whether every route in the solution respects vehicle capacity by summing customer demands for each route. Returns True only if all routes are feasible.

In [17]:
def check_solution_feasibility(solution, demands, capacity):
    """
    solution: list of routes (each route includes depot 0)
    """
    print("\n============================")
    print("CHECKING FULL SOLUTION FEASIBILITY")
    print("============================")
    print("Vehicle capacity:", capacity)
    print("Number of routes:", len(solution))
    print("------------------------------------")

    route_index = 1
    for route in solution:
        print(f"\n--- Checking Route {route_index} ---")
        print("Original route (with depot):", route)

        # Remove depot 0 to compute load
        route_customers = [c for c in route if c != 0]
        print("Customers in route (no depot):", route_customers)

        feasible = is_capacity_feasible(route_customers, demands, capacity)

        if not feasible:
            print(f" Route {route_index} is NOT feasible (exceeds capacity)")
            return False
        else:
            print(f" Route {route_index} is feasible")

        route_index += 1

    print("\nALL ROUTES ARE FEASIBLE")
    return True


Converts a GA chromosome (giant tour) into multiple capacity-feasible routes by splitting whenever adding a customer would exceed vehicle capacity.

In [19]:
def decode_giant_tour(chromosome, demands, capacity, depot=0):
    """
    Decodes giant tour (like a GA chromosome) into multiple capacity-feasible routes.
    Adds debug print statements.
    """
    print("\n=======================================")
    print("DECODING GIANT TOUR INTO ROUTES")
    print("=======================================")
    print("Chromosome:", chromosome)
    print("Vehicle capacity:", capacity)
    print("---------------------------------------")

    routes = []
    current_route = [depot]
    current_load = 0
    route_num = 1

    for cust in chromosome:
        demand = demands[cust]
        print(f"\nConsidering customer {cust} (Demand = {demand})")
        print(f"Current route load = {current_load}, Remaining capacity = {capacity - current_load}")

        # If adding the customer exceeds capacity → start new route
        if current_load + demand > capacity:
            print(f"⚠️ Adding customer {cust} exceeds capacity. Closing Route {route_num}.")
            current_route.append(depot)
            print(f"Route {route_num} finalized:", current_route)

            routes.append(current_route)
            route_num += 1

            # Start new route
            current_route = [depot]
            current_load = 0
            print(f"Starting Route {route_num}...")

        # Add customer to current route
        current_route.append(cust)
        current_load += demand
        print(f"Added customer {cust} → new load = {current_load}")
        print("Current route:", current_route)

    # Close final route
    current_route.append(depot)
    routes.append(current_route)

    print(f"\nFinal Route {route_num}:", current_route)

    print("\n=======================================")
    print("FINAL DECODED ROUTES:")
    for i, r in enumerate(routes):
        print(f"Route {i+1}: {r}")
    print("=======================================\n")

    return routes


Computes the distance of each route by summing the pairwise distances between consecutive customers, printing each leg’s contribution. Then computes the total solution distance by adding all route distances with detailed route-by-route output.

In [18]:
def route_distance(route, dist):
    """
    route: [0, 4, 7, 0]
    Computes distance of a single route.
    Includes detailed print statements.
    """
    print("\n-----------------------------------------")
    print("Computing distance for route:", route)
    cost = 0

    for i in range(len(route) - 1):
        a = route[i]
        b = route[i + 1]
        d = dist[a, b]
        cost += d
        print(f"Leg {a} → {b} : distance = {d:.3f}, cumulative = {cost:.3f}")

    print("Total distance for this route:", cost)
    print("-----------------------------------------\n")
    return cost



def total_solution_distance(solution, dist):
    """
    Computes total distance of all routes.
    Includes detailed print statements.
    """
    print("\n=========================================")
    print("COMPUTING TOTAL SOLUTION DISTANCE")
    print("=========================================")
    total = 0

    for idx, route in enumerate(solution, start=1):
        print(f"\n### Route {idx} ###")
        route_cost = route_distance(route, dist)
        total += route_cost
        print(f"Route {idx} cost = {route_cost:.3f}")
        print(f"Cumulative total = {total:.3f}")

    print("\n=========================================")
    print("FINAL TOTAL DISTANCE:", total)
    print("=========================================\n")
    return total


Loads depot and customer coordinates from the MDRP dataset, separating depot rows from customer rows and assigning unit demands since no demand data exists in the file. Prints detailed diagnostics to verify the extracted depots, customers, and generated demand values.

In [21]:
import numpy as np
import pandas as pd
from math import sqrt

def load_mdrp_format(path="19MDVRP Problem Sets.xlsx", sheet="Problem 7"):
    print("========================================")
    print(" LOADING MDRP / CVRP DATA")
    print("========================================")
    print(f"File: {path}")
    print(f"Sheet: {sheet}")

    # Load sheet
    df = pd.read_excel(path, sheet_name=sheet)

    print("\n--- Raw DataFrame Head ---")
    print(df.head())

    print("\n--- Columns Found ---")
    print(df.columns.tolist())

    # --- Depot rows ---
    depot_df = df[df['Depot x coordinate'].notna()]
    depots = depot_df[['Depot x coordinate', 'Depot y coordinate']].to_numpy()

    print("\n--- Depot Rows Found ---")
    print(depot_df)

    print("\nDepot Coordinates Extracted:")
    print(depots)

    # --- Customer rows ---
    cust_df = df[df['Customer Number'].notna()]
    customers = cust_df[['x coordinate', 'y coordinate']].to_numpy()

    print("\n--- Customer Rows Found ---")
    print(cust_df)

    print("\nCustomer Coordinates Extracted:")
    print(customers)

    # --- DEMANDS (missing in sheet) ---
    demands = np.ones(len(customers), dtype=int)

    print("\nCustomer Demands Assigned:")
    print(demands)

    print("\n========================================")
    print(" DATA LOADED SUCCESSFULLY")
    print("========================================\n")

    return depots, customers, demands


Calls the MDRP data loader to extract depot coordinates, customer coordinates, and generated demand values from the dataset. Prints the loaded depots, customer locations, and a preview of the assigned demands for verification.

In [13]:
depots, coords, demands = load_mdrp_format("19MDVRP Problem Sets.xlsx", sheet="Problem 7")

print("Depots:\n", depots)
print("Customer coords:\n", coords)
print("Demands:\n", demands[:10])


Depots:
 [[15. 35.]
 [55. 35.]
 [35. 20.]
 [35. 50.]]
Customer coords:
 [[41 49]
 [35 17]
 [55 45]
 [55 20]
 [15 30]
 [25 30]
 [20 50]
 [10 43]
 [55 60]
 [30 60]
 [20 65]
 [50 35]
 [30 25]
 [15 10]
 [30  5]
 [10 20]
 [ 5 30]
 [20 40]
 [15 60]
 [45 65]
 [45 20]
 [45 10]
 [55  5]
 [65 35]
 [65 20]
 [45 30]
 [35 40]
 [41 37]
 [64 42]
 [40 60]
 [31 52]
 [35 69]
 [53 52]
 [65 55]
 [63 65]
 [ 2 60]
 [20 20]
 [ 5  5]
 [60 12]
 [40 25]
 [42  7]
 [24 12]
 [23  3]
 [11 14]
 [ 6 38]
 [ 2 48]
 [ 8 56]
 [13 52]
 [ 6 68]
 [47 47]
 [49 58]
 [27 43]
 [37 31]
 [57 29]
 [63 23]
 [53 12]
 [32 12]
 [36 26]
 [21 24]
 [17 34]
 [12 24]
 [24 58]
 [27 69]
 [15 77]
 [62 77]
 [49 73]
 [67  5]
 [56 39]
 [37 47]
 [37 56]
 [57 68]
 [47 16]
 [44 17]
 [46 13]
 [49 11]
 [49 42]
 [53 43]
 [61 52]
 [57 48]
 [56 37]
 [55 54]
 [15 47]
 [14 37]
 [11 31]
 [16 22]
 [ 4 18]
 [28 18]
 [26 52]
 [26 35]
 [31 67]
 [15 19]
 [22 22]
 [18 24]
 [26 27]
 [25 24]
 [22 27]
 [25 21]
 [19 21]
 [20 26]
 [18 18]]
Demands:
 [1 1 1 1 1 1 1 1 